# LoRA from scratch: build it, prove the win, prove the merge is free

Full fine-tuning a 7B model means storing a gradient and Adam's optimizer state for **all** 7
billion parameters — a training footprint of ~100+ GB, and a fresh 14 GB checkpoint per task.
**LoRA** (Low-Rank Adaptation) refuses: it **freezes** the pretrained weight `W` and trains only a
tiny low-rank update `ΔW = (α/r)·B·A`. You update <1% of the parameters, and at inference you fold
`B·A` back into `W` for **zero added latency**.

This notebook builds a LoRA-wrapped linear layer from scratch in small pieces. **By the end you'll
have built the adapter, printed the trainable-vs-frozen parameter counts (a 64× reduction),
confirmed gradients reach `A,B` but never the frozen `W`, watched `ΔW = 0` at init, trained the
adapter, and proven the merged single-matrix forward equals the two-path LoRA forward.**

## Setup: imports, seed, and honest device reporting

We detect the best accelerator but **pin the reproducible trace to CPU** so the printed numbers are
bit-stable across machines. We print the detected device honestly, the torch version, and seed for
reproducibility. Every tensor below is created on `DEVICE` (= CPU here).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
torch.manual_seed(SEED)
torch.set_num_threads(1)  # single-threaded CPU -> sweep losses are bit-reproducible and match lora_peft.py exactly

# Detect the best accelerator, but pin the trace to CPU for reproducible printed numbers.
DETECTED = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
DEVICE = "cpu"  # reproducible trace on CPU regardless of what is detected
print(f"device: {DEVICE} (detected {DETECTED}; pinned to CPU for reproducibility)")
print("torch:", torch.__version__)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0


## The hyperparameters — the knobs LoRA is about

`r` (rank) is the low-rank bottleneck, `r ≪ d` — the whole point. `alpha` is a fixed scaling
constant; the update is multiplied by `alpha/r` so changing `r` doesn't force you to re-tune the
learning rate. We use a square `d = 1024` layer so the `d²` vs `2rd` contrast is clean and the
reduction comes out to exactly `d/2r = 64×`.

In [2]:
IN_FEATURES = 1024   # d: input width
OUT_FEATURES = 1024  # k: output width (square -> d/2r is a clean ratio)
LORA_RANK = 8        # r: the low-rank bottleneck, r << d
LORA_ALPHA = 16      # alpha: scaling numerator; update *= alpha/r
TRUE_RANK = 4        # the synthetic target update is genuinely rank-4 (LoRA's premise)
print(f"d={IN_FEATURES}, r={LORA_RANK}, alpha={LORA_ALPHA}  ->  scaling alpha/r = {LORA_ALPHA/LORA_RANK}")

d=1024, r=8, alpha=16  ->  scaling alpha/r = 2.0


## Build the LoRA layer from scratch

The layer holds three tensors: the **frozen** pretrained weight `W` (`requires_grad=False`, so the
optimizer never touches it), the **random-init** down-projection `A` (`r×d`), and the **zero-init**
up-projection `B` (`d×r`). Because `B = 0`, the product `B·A = 0` at step 0, so `ΔW = 0` and the
adapted layer is exactly the pretrained one. The forward sums the frozen path `x·Wᵀ` and the adapter
path `(α/r)·B·A·x`.

In [3]:
class LoRALinear(nn.Module):
    def __init__(self, in_f, out_f, rank, alpha, device=DEVICE):
        super().__init__()
        self.scaling = alpha / rank                          # the alpha/r scale
        self.weight = nn.Parameter(
            torch.randn(out_f, in_f, device=device) * 0.02,
            requires_grad=False,                              # W FROZEN -- only A, B train
        )
        self.lora_a = nn.Parameter(torch.empty(rank, in_f, device=device))   # A: r x d, random
        self.lora_b = nn.Parameter(torch.zeros(out_f, rank, device=device))  # B: d x r, ZERO -> dW=0
        nn.init.kaiming_uniform_(self.lora_a, a=5**0.5)

    def forward(self, x):
        base = F.linear(x, self.weight)                                      # x . W^T  (frozen)
        update = F.linear(F.linear(x, self.lora_a), self.lora_b) * self.scaling  # (alpha/r) . B . A . x
        return base + update

    def merged_weight(self):
        return self.weight + (self.lora_b @ self.lora_a) * self.scaling      # W + (alpha/r) . B . A

layer = LoRALinear(IN_FEATURES, OUT_FEATURES, rank=LORA_RANK, alpha=LORA_ALPHA)
print(layer)

LoRALinear()


## The headline win: trainable vs frozen parameters

Full fine-tuning would train all `d²` parameters of `W`. LoRA trains only `A` (`r×d`) and `B`
(`d×r`) — `2rd` parameters. We count both and print the reduction ratio `d/2r`. For `d=1024, r=8`
that's `1024/16 = 64×` fewer trainable parameters.

In [4]:
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
full_ft = IN_FEATURES * OUT_FEATURES  # full fine-tuning would update all d*k params
print(f"full fine-tuning (all of W):  {full_ft:>12,} params  (d^2)")
print(f"LoRA trainable (A + B):       {trainable:>12,} params  (2*r*d, r={LORA_RANK})")
print(f"frozen (W, untouched):        {frozen:>12,} params")
print(f"reduction: {full_ft/trainable:.0f}x fewer ({100*trainable/full_ft:.2f}% of full FT)")

full fine-tuning (all of W):     1,048,576 params  (d^2)
LoRA trainable (A + B):             16,384 params  (2*r*d, r=8)
frozen (W, untouched):           1,048,576 params
reduction: 64x fewer (1.56% of full FT)


## `ΔW = 0` at initialization (because `B = 0`)

The zero-init of `B` is not a detail — it's *why training starts at the pretrained model*. We compute
`B·A` directly and confirm its largest entry is exactly zero before any training. If we'd zeroed `A`
instead, `A`'s gradient would also be zero and it could never start learning; zeroing `B` keeps `A`'s
gradient alive.

In [5]:
delta_at_init = (layer.lora_b @ layer.lora_a).abs().max().item()
print(f"max|dW| before any training: {delta_at_init:.1e}  (exactly 0 -> starts at the pretrained model)")

max|dW| before any training: 0.0e+00  (exactly 0 -> starts at the pretrained model)


## A tiny synthetic task whose true update is low-rank

To make LoRA's premise concrete, we build a regression task where the *change* from the base weight
is genuinely **rank-4**: `y = x·(W + ΔW*)ᵀ` with `ΔW*` the product of two thin matrices (rank
`TRUE_RANK`). A LoRA of rank `>= 4` can fit this exactly; a smaller rank cannot. We copy this same
frozen base into the layer so the adapter has the right target to learn.

In [6]:
torch.manual_seed(SEED)
N_SAMPLES = 256
base_weight = torch.randn(OUT_FEATURES, IN_FEATURES, device=DEVICE) * 0.02
u = torch.randn(OUT_FEATURES, TRUE_RANK, device=DEVICE) * 0.1
v = torch.randn(TRUE_RANK, IN_FEATURES, device=DEVICE) * 0.1
delta_star = u @ v                       # a genuinely rank-TRUE_RANK update
x = torch.randn(N_SAMPLES, IN_FEATURES, device=DEVICE)
y = F.linear(x, base_weight + delta_star)  # ground truth uses the adapted weight
layer.weight.data.copy_(base_weight)       # adapter must learn delta_star on top of this base
print(f"x: {tuple(x.shape)}   y: {tuple(y.shape)}   true update rank: {TRUE_RANK}")

x: (256, 1024)   y: (256, 1024)   true update rank: 4


## Gradients reach `A` and `B` only — never the frozen `W`

One backward pass, then we inspect `.grad` on each tensor. `A` and `B` get gradients; the frozen `W`
gets `None`. This is the mechanical reason LoRA's optimizer state is tiny — it only exists for the
parameters that require grad.

In [7]:
layer.zero_grad(set_to_none=True)
loss = F.mse_loss(layer(x), y)
loss.backward()
print(f"A receives grad: {layer.lora_a.grad is not None}")
print(f"B receives grad: {layer.lora_b.grad is not None}")
print(f"frozen W receives grad: {layer.weight.grad is not None}  (expected False)")
assert layer.lora_a.grad is not None and layer.lora_b.grad is not None
assert layer.weight.grad is None, "W must stay frozen"

A receives grad: True
B receives grad: True
frozen W receives grad: False  (expected False)


## Train the adapter (frozen `W`, trainable `A,B`)

We hand the optimizer **only** the parameters that require grad — i.e. `A` and `B`, never `W`. The
loss should fall by many orders of magnitude as the rank-8 adapter learns the rank-4 update.

In [8]:
trainable_params = [p for p in layer.parameters() if p.requires_grad]  # A, B only
optimizer = torch.optim.Adam(trainable_params, lr=1e-2)
losses = []
for _ in range(300):
    optimizer.zero_grad()
    loss = F.mse_loss(layer(x), y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f"loss: {losses[0]:.4e} (step 0) -> {losses[-1]:.4e} (step {len(losses)})  [{losses[0]/losses[-1]:.0f}x lower]")

loss: 3.8875e-01 (step 0) -> 2.1496e-07 (step 300)  [1808468x lower]


## The merge: prove zero added inference latency

Both the frozen path and the adapter path are linear in `x`, so
`h = W·x + (α/r)·B·A·x = (W + (α/r)·B·A)·x`. We compute the merged weight **once**, run a single
matmul with it, and assert the output equals the two-path LoRA forward to floating-point tolerance.
That equality *is* the zero-latency proof: at inference the adapter vanishes into the weights.

In [9]:
unmerged = layer(x)                              # two paths: frozen + adapter
merged = F.linear(x, layer.merged_weight())      # one fused matmul with W + (alpha/r).B.A
max_diff = (unmerged - merged).abs().max().item()
ok = torch.allclose(unmerged, merged, atol=1e-4) # 1e-4 sits 4+ orders below any real merge bug; the ~8e-6 gap is fp32 accumulation noise
note = "bitwise-identical" if max_diff == 0 else "within float-noise tolerance"
print(f"merged == unmerged forward: {ok}   max abs diff: {max_diff:.2e}  ({note})")
assert ok, "merged weight must reproduce the unmerged LoRA forward"

merged == unmerged forward: True   max abs diff: 8.11e-06  (within float-noise tolerance)


## The rank cliff: params (linear in `r`) vs fit (cliffs at the true rank)

Finally, the central tradeoff. We sweep `r` and, for each, report the trainable params (`2rd`, linear
in `r`) and the final fit loss. On a task whose true update is rank-4, the loss falls off a cliff
exactly at `r = 4` and is flat afterward — **more rank past the true rank only buys parameters, never
quality**; below it, the adapter underfits. This is why `r = 8` or `16` is usually plenty for real
fine-tunes.

In [10]:
# Use the SAME canonical sweep from lora_peft.py so the notebook, the script, and the figure
# print one identical set of numbers (the reproducibility rule: page == notebook == .py == figure).
# rank_sweep() re-seeds before each rank, so it is independent of anything run above.
from lora_peft import rank_sweep, IN_FEATURES as D, OUT_FEATURES as K

full_ft = D * K
print(f"{'rank r':>7} | {'trainable':>10} | {'% of full FT':>12} | {'final loss':>12}")
print('-' * 52)
for r, params, final in rank_sweep(x, y):
    print(f"{r:>7} | {params:>10,} | {100*params/full_ft:>11.3f}% | {final:>12.4e}")

 rank r |  trainable | % of full FT |   final loss
----------------------------------------------------


      1 |      2,048 |       0.195% |   2.7211e-01
      2 |      4,096 |       0.391% |   1.7230e-01
      4 |      8,192 |       0.781% |   8.5932e-07
      8 |     16,384 |       1.562% |   2.1496e-07
     16 |     32,768 |       3.125% |   1.0558e-07
     32 |     65,536 |       6.250% |   1.3950e-07


## What we proved

- **The win:** a LoRA layer trains `2rd` parameters instead of `d²` — here **64× fewer** — because
  `W` is frozen and only the thin `A,B` get gradients and optimizer state.
- **The init:** `B = 0` makes `ΔW = 0` at step 0, so training starts *exactly* at the pretrained model.
- **The merge:** because both paths are linear, `W + (α/r)·B·A` is a single matrix whose forward is
  identical to the two-path LoRA forward — **zero added inference latency**.
- **The knob:** rank trades parameters against fit, and the sweet spot is near the update's true
  (low) intrinsic rank — which is why small `r` works.

The same verified code lives in [`lora_peft.py`](lora_peft.py); the figures on the page are generated
from these exact numbers by [`make_figures_12.py`](make_figures_12.py). See the
[concept page](../12-LoRA-and-PEFT.md) for QLoRA, the wider PEFT family, and production details.